In [7]:
import time
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import yfinance as yf

def classify_vix_regime(
    vix_data: pd.DataFrame,
    lookback_days: int = 25,
    threshold_pct: float = 0.10,
) -> pd.DataFrame:
    vix_close = vix_data["Close"].rename("VIX_Close")
    prior_25d_avg = vix_close.shift(1).rolling(
        lookback_days,
        min_periods=lookback_days,
    ).mean()
    high_vol_threshold = (1 + threshold_pct) * prior_25d_avg

    return pd.DataFrame(
        {
            "VIX_Close": vix_close,
            "Prior_25D_Avg": prior_25d_avg,
            "High_Vol_Threshold": high_vol_threshold,
            "Regime": np.where(
                prior_25d_avg.notna() & (vix_close > high_vol_threshold),
                "high_vol",
                "low_vol",
            ),
        }
    )


DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

start_date = "1980-01-01"
end_date = "2025-12-31"

# Yahoo's end date is exclusive, so use 2026-01-01 to include all of 2025.
vix = yf.download(
    "^VIX",
    start=start_date,
    end="2026-01-01",
    auto_adjust=False,
    progress=False,
)

if vix.empty:
    raise ValueError("No VIX data was returned from Yahoo Finance.")

if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.get_level_values(0)

vix_regimes = classify_vix_regime(vix)
vix_regimes.to_csv(DATA_DIR / "vix_regime_classification.csv", index_label="Date")

print(f"Requested range: {start_date} to {end_date}")
print(f"Returned range: {vix.index.min().date()} to {vix.index.max().date()}")
print("Saved: data/vix_regime_classification.csv")
vix_regimes.head(), vix_regimes.tail()


Requested range: 1980-01-01 to 2025-12-31
Returned range: 1990-01-02 to 2025-12-31
Saved: data/vix_regime_classification.csv


(            VIX_Close  Prior_25D_Avg  High_Vol_Threshold   Regime
 Date                                                             
 1990-01-02  17.240000            NaN                 NaN  low_vol
 1990-01-03  18.190001            NaN                 NaN  low_vol
 1990-01-04  19.219999            NaN                 NaN  low_vol
 1990-01-05  20.110001            NaN                 NaN  low_vol
 1990-01-08  20.260000            NaN                 NaN  low_vol,
             VIX_Close  Prior_25D_Avg  High_Vol_Threshold   Regime
 Date                                                             
 2025-12-24      13.47        17.6932            19.46252  low_vol
 2025-12-26      13.60        17.2444            18.96884  low_vol
 2025-12-29      14.20        16.8420            18.52620  low_vol
 2025-12-30      14.33        16.3532            17.98852  low_vol
 2025-12-31      14.95        15.9892            17.58812  low_vol)

In [8]:
from pathlib import Path

# -----------------------------
# 1) Define candidate universes
# -----------------------------
# Large caps: liquid, well-known names
LARGE_CAPS = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "BRK-B", "JPM", "LLY", "AVGO",
    "XOM", "UNH", "V", "MA", "COST", "PG", "JNJ", "HD", "ABBV", "BAC",
    "KO", "PEP", "MRK", "WMT", "CVX", "ADBE", "NFLX", "CRM", "AMD", "TMO"
]

# Smaller caps / mid-small blend for practicality on Yahoo
# You can replace with a Russell 2000 subset if you already have tickers elsewhere.
SMALL_CAPS = [
    "CROX", "AAON", "UFPI", "FIVE", "MTH", "SIGI", "GBCI", "BANF", "INDB", "PSMT",
    "SLAB", "BCPC", "COLB", "HOMB", "RNST", "IBP", "LFUS", "FULT", "WDFC", "CBRL",
    "CVCO", "TCBK", "FRME", "SXI", "NMIH", "FHI", "NSIT", "OMCL", "HLI", "IPAR"
]

ALL_TICKERS = LARGE_CAPS + SMALL_CAPS


# ---------------------------------------------------------
# 2) Download static info and keep only sufficiently liquid
# ---------------------------------------------------------
def get_ticker_info(ticker: str) -> Dict:
    """
    Fetch basic ticker info from Yahoo via yfinance.
    """
    try:
        tk = yf.Ticker(ticker)
        info = tk.info  # may occasionally fail / be incomplete
        return {
            "ticker": ticker,
            "marketCap": info.get("marketCap"),
            "averageVolume": info.get("averageVolume"),
            "averageVolume10days": info.get("averageVolume10days"),
            "quoteType": info.get("quoteType"),
            "exchange": info.get("exchange"),
            "sector": info.get("sector"),
            "longName": info.get("longName"),
        }
    except Exception:
        return {
            "ticker": ticker,
            "marketCap": np.nan,
            "averageVolume": np.nan,
            "averageVolume10days": np.nan,
            "quoteType": None,
            "exchange": None,
            "sector": None,
            "longName": None,
        }


def build_info_table(tickers: List[str], sleep_sec: float = 0.2) -> pd.DataFrame:
    rows = []
    for t in tickers:
        rows.append(get_ticker_info(t))
        time.sleep(sleep_sec)  # reduce risk of throttling
    df = pd.DataFrame(rows)
    # Prefer 10-day avg volume if available
    df["avg_vol_used"] = df["averageVolume10days"].fillna(df["averageVolume"])
    return df


def filter_liquid_universe(
    info_df: pd.DataFrame,
    min_market_cap: float,
    min_avg_volume: float
) -> pd.DataFrame:
    """
    Keep liquid stocks only.
    """
    out = info_df.copy()
    out = out[
        (out["marketCap"].fillna(0) >= min_market_cap) &
        (out["avg_vol_used"].fillna(0) >= min_avg_volume)
    ].copy()
    return out.sort_values(["marketCap", "avg_vol_used"], ascending=False)


# ---------------------------------------------------
# 3) Download price history with and without dividends
# ---------------------------------------------------
def download_price_panels(
    tickers: List[str],
    start: str | None = None,
    end: str | None = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns two wide price panels:
      - prices_no_div: unadjusted Close (no dividends reinvested)
      - prices_with_div: dividend/split-adjusted Close
    """

    if start is None:
        start = globals().get("start_date", "1980-01-01")

    # Unadjusted: includes OHLC + Adj Close when auto_adjust=False
    raw = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        interval="1d",
        auto_adjust=False,
        actions=True,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    # Adjusted: prices adjusted for dividends and splits
    adj = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        interval="1d",
        auto_adjust=True,
        actions=False,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    # Handle single-ticker edge case
    if len(tickers) == 1:
        t = tickers[0]
        prices_no_div = raw["Close"].to_frame(name=t)
        prices_with_div = adj["Close"].to_frame(name=t)
    else:
        prices_no_div = pd.concat(
            {t: raw[t]["Close"] for t in tickers if t in raw.columns.get_level_values(0)},
            axis=1
        )
        prices_with_div = pd.concat(
            {t: adj[t]["Close"] for t in tickers if t in adj.columns.get_level_values(0)},
            axis=1
        )

    # Clean columns
    prices_no_div.columns = prices_no_div.columns.get_level_values(0) if isinstance(prices_no_div.columns, pd.MultiIndex) else prices_no_div.columns
    prices_with_div.columns = prices_with_div.columns.get_level_values(0) if isinstance(prices_with_div.columns, pd.MultiIndex) else prices_with_div.columns

    return prices_no_div.sort_index(), prices_with_div.sort_index()


def compute_returns(price_panel: pd.DataFrame, log_returns: bool = True) -> pd.DataFrame:
    if log_returns:
        return np.log(price_panel / price_panel.shift(1))
    return price_panel.pct_change()


# -----------------------------------------
# 4) Main pipeline: build liquid portfolios
# -----------------------------------------
if __name__ == "__main__":
    info_df = build_info_table(ALL_TICKERS)

    # Example liquidity filters:
    # - Large caps: market cap >= $10bn, avg volume >= 2m shares/day
    # - Small caps: market cap between $300m and $5bn, avg volume >= 300k shares/day
    # You can tighten / relax these depending on how many names you want.

    large_info = info_df[
        (info_df["ticker"].isin(LARGE_CAPS)) &
        (info_df["marketCap"].fillna(0) >= 10e9) &
        (info_df["avg_vol_used"].fillna(0) >= 2_000_000)
    ].copy()

    small_info = info_df[
        (info_df["ticker"].isin(SMALL_CAPS)) &
        (info_df["marketCap"].fillna(0).between(300e6, 5e9)) &
        (info_df["avg_vol_used"].fillna(0) >= 300_000)
    ].copy()

    large_tickers = large_info["ticker"].tolist()
    small_tickers = small_info["ticker"].tolist()

    print("Selected liquid large caps:", large_tickers)
    print("Selected liquid small caps:", small_tickers)

    # Download data
    data_dir = globals().get("DATA_DIR", Path("data"))
    data_dir.mkdir(exist_ok=True)
    portfolio_start_date = globals().get("start_date", "1980-01-01")
    large_no_div, large_with_div = download_price_panels(large_tickers, start=portfolio_start_date)
    small_no_div, small_with_div = download_price_panels(small_tickers, start=portfolio_start_date)

    # Compute returns
    large_rets_no_div = compute_returns(large_no_div)
    large_rets_with_div = compute_returns(large_with_div)

    small_rets_no_div = compute_returns(small_no_div)
    small_rets_with_div = compute_returns(small_with_div)

    # Equal-weight portfolio returns (simple example)
    ew_large_no_div = large_rets_no_div.mean(axis=1).rename("EW_Large_NoDiv")
    ew_large_with_div = large_rets_with_div.mean(axis=1).rename("EW_Large_WithDiv")

    ew_small_no_div = small_rets_no_div.mean(axis=1).rename("EW_Small_NoDiv")
    ew_small_with_div = small_rets_with_div.mean(axis=1).rename("EW_Small_WithDiv")

    portfolio_returns = pd.concat(
        [ew_large_no_div, ew_large_with_div, ew_small_no_div, ew_small_with_div],
        axis=1
    )

    # Save outputs
    info_df.to_csv(data_dir / "universe_info.csv", index=False)

    large_no_div.to_csv(data_dir / "large_caps_prices_no_dividends.csv")
    large_with_div.to_csv(data_dir / "large_caps_prices_with_dividends.csv")
    small_no_div.to_csv(data_dir / "small_caps_prices_no_dividends.csv")
    small_with_div.to_csv(data_dir / "small_caps_prices_with_dividends.csv")

    large_rets_no_div.to_csv(data_dir / "large_caps_returns_no_dividends.csv")
    large_rets_with_div.to_csv(data_dir / "large_caps_returns_with_dividends.csv")
    small_rets_no_div.to_csv(data_dir / "small_caps_returns_no_dividends.csv")
    small_rets_with_div.to_csv(data_dir / "small_caps_returns_with_dividends.csv")

    portfolio_returns.to_csv(data_dir / "portfolio_returns_equal_weight.csv")

    print("\nSaved:")
    print("- data/universe_info.csv")
    print("- data/large/small caps prices and returns, with and without dividends")
    print("- data/portfolio_returns_equal_weight.csv")

Selected liquid large caps: ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'BRK-B', 'JPM', 'LLY', 'AVGO', 'XOM', 'UNH', 'V', 'MA', 'PG', 'JNJ', 'HD', 'ABBV', 'BAC', 'KO', 'PEP', 'MRK', 'WMT', 'CVX', 'ADBE', 'NFLX', 'CRM', 'AMD', 'TMO']
Selected liquid small caps: ['CROX', 'MTH', 'SIGI', 'INDB', 'RNST', 'FULT', 'CBRL', 'FRME', 'NMIH', 'FHI', 'NSIT', 'OMCL']

Saved:
- data/universe_info.csv
- data/large/small caps prices and returns, with and without dividends
- data/portfolio_returns_equal_weight.csv


In [9]:
import numpy as np
import pandas as pd

from pathlib import Path


def load_portfolio_return_series() -> pd.Series:
    data_dir = globals().get("DATA_DIR", Path("data"))
    if "portfolio_returns" in globals():
        returns_df = portfolio_returns.copy()
    else:
        returns_df = pd.read_csv(
            data_dir / "portfolio_returns_equal_weight.csv",
            index_col="Date",
            parse_dates=True,
        )

    preferred_cols = [
        col for col in ["EW_Large_WithDiv", "EW_Small_WithDiv"] if col in returns_df.columns
    ]
    if preferred_cols:
        portfolio_series = returns_df[preferred_cols].mean(axis=1)
    else:
        numeric_cols = returns_df.select_dtypes(include=[np.number]).columns.tolist()
        if not numeric_cols:
            raise ValueError("No numeric portfolio return columns were found.")
        portfolio_series = returns_df[numeric_cols].mean(axis=1)

    return portfolio_series.dropna().rename("Portfolio_Return")


def load_vix_feature_table() -> pd.DataFrame:
    data_dir = globals().get("DATA_DIR", Path("data"))
    if "vix_regimes" in globals():
        vix_features = vix_regimes.copy()
    elif (data_dir / "vix_regime_classification.csv").exists():
        vix_features = pd.read_csv(
            data_dir / "vix_regime_classification.csv",
            index_col="Date",
            parse_dates=True,
        )
    elif "vix" in globals():
        vix_features = classify_vix_regime(vix)
    else:
        raise FileNotFoundError(
            "Run the VIX cell first or make sure data/vix_regime_classification.csv exists."
        )

    return vix_features[["VIX_Close"]].copy()


def build_hmm_feature_table(realized_vol_window: int = 21) -> pd.DataFrame:
    portfolio_return = load_portfolio_return_series()
    vix_feature = load_vix_feature_table()

    feature_table = pd.concat(
        [
            portfolio_return,
            portfolio_return.abs().rename("Abs_Return"),
            portfolio_return.rolling(realized_vol_window).std(ddof=0).mul(np.sqrt(252)).rename(
                "Realized_Volatility"
            ),
            vix_feature["VIX_Close"],
        ],
        axis=1,
    ).dropna()

    if feature_table.empty:
        raise ValueError("No aligned HMM features are available after dropping missing values.")

    return feature_table


def logsumexp(values: np.ndarray, axis: int | None = None, keepdims: bool = False) -> np.ndarray:
    max_values = np.max(values, axis=axis, keepdims=True)
    stable_values = values - max_values
    summed = np.sum(np.exp(stable_values), axis=axis, keepdims=True)
    output = max_values + np.log(summed)

    if not keepdims:
        output = np.squeeze(output, axis=axis)

    return output


def initialize_hmm_parameters(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n_obs, n_features = X.shape
    split_order = np.argsort(X[:, [1, 2, 3]].mean(axis=1))
    initial_states = np.zeros(n_obs, dtype=int)
    initial_states[split_order[n_obs // 2 :]] = 1

    means = []
    variances = []
    for state in range(2):
        state_obs = X[initial_states == state]
        if len(state_obs) == 0:
            state_obs = X
        means.append(state_obs.mean(axis=0))
        variances.append(np.clip(state_obs.var(axis=0), 1e-4, None))

    startprob = np.array([0.5, 0.5], dtype=float)
    transmat = np.array([[0.95, 0.05], [0.05, 0.95]], dtype=float)

    return startprob, transmat, np.vstack(means), np.vstack(variances)


def gaussian_logpdf_diag(X: np.ndarray, means: np.ndarray, variances: np.ndarray) -> np.ndarray:
    log_det = np.sum(np.log(2 * np.pi * variances), axis=1)
    squared_mahalanobis = np.sum(
        ((X[:, None, :] - means[None, :, :]) ** 2) / variances[None, :, :],
        axis=2,
    )
    return -0.5 * (log_det[None, :] + squared_mahalanobis)


def forward_backward(
    log_startprob: np.ndarray,
    log_transmat: np.ndarray,
    emission_log_prob: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, float]:
    n_obs, n_states = emission_log_prob.shape

    log_alpha = np.empty((n_obs, n_states))
    log_alpha[0] = log_startprob + emission_log_prob[0]
    for t in range(1, n_obs):
        log_alpha[t] = emission_log_prob[t] + logsumexp(
            log_alpha[t - 1][:, None] + log_transmat,
            axis=0,
        )

    log_beta = np.zeros((n_obs, n_states))
    for t in range(n_obs - 2, -1, -1):
        log_beta[t] = logsumexp(
            log_transmat + emission_log_prob[t + 1][None, :] + log_beta[t + 1][None, :],
            axis=1,
        )

    log_likelihood = float(logsumexp(log_alpha[-1], axis=0))
    log_gamma = log_alpha + log_beta - log_likelihood
    gamma = np.exp(log_gamma)
    gamma /= gamma.sum(axis=1, keepdims=True)

    xi = np.empty((n_obs - 1, n_states, n_states))
    for t in range(n_obs - 1):
        log_xi_t = (
            log_alpha[t][:, None]
            + log_transmat
            + emission_log_prob[t + 1][None, :]
            + log_beta[t + 1][None, :]
            - log_likelihood
        )
        xi[t] = np.exp(log_xi_t)
        xi[t] /= xi[t].sum()

    return gamma, xi, log_likelihood


def fit_gaussian_hmm(
    X: np.ndarray,
    max_iter: int = 200,
    tol: float = 1e-4,
    min_covar: float = 1e-4,
) -> dict:
    startprob, transmat, means, variances = initialize_hmm_parameters(X)
    previous_log_likelihood = -np.inf

    for iteration in range(max_iter):
        log_startprob = np.log(np.clip(startprob, 1e-12, None))
        log_transmat = np.log(np.clip(transmat, 1e-12, None))
        emission_log_prob = gaussian_logpdf_diag(X, means, variances)
        gamma, xi, log_likelihood = forward_backward(
            log_startprob,
            log_transmat,
            emission_log_prob,
        )

        gamma_sum = np.clip(gamma.sum(axis=0), 1e-12, None)
        startprob = np.clip(gamma[0], 1e-12, None)
        startprob /= startprob.sum()

        transmat = np.clip(xi.sum(axis=0), 1e-12, None)
        transmat /= transmat.sum(axis=1, keepdims=True)

        means = (gamma.T @ X) / gamma_sum[:, None]
        centered = X[:, None, :] - means[None, :, :]
        variances = (gamma[:, :, None] * centered**2).sum(axis=0) / gamma_sum[:, None]
        variances = np.clip(variances, min_covar, None)

        if np.abs(log_likelihood - previous_log_likelihood) < tol:
            break

        previous_log_likelihood = log_likelihood

    return {
        "startprob": startprob,
        "transmat": transmat,
        "means": means,
        "variances": variances,
        "gamma": gamma,
        "log_likelihood": log_likelihood,
        "iterations": iteration + 1,
    }


portfolio_hmm_features = build_hmm_feature_table(realized_vol_window=21)
feature_means = portfolio_hmm_features.mean()
feature_stds = portfolio_hmm_features.std(ddof=0).replace(0, 1.0)
X_scaled = ((portfolio_hmm_features - feature_means) / feature_stds).to_numpy()

portfolio_hmm_model = fit_gaussian_hmm(X_scaled)

state_feature_means = pd.DataFrame(
    (
        portfolio_hmm_model["gamma"].T @ portfolio_hmm_features.to_numpy()
    ) / portfolio_hmm_model["gamma"].sum(axis=0)[:, None],
    index=[0, 1],
    columns=portfolio_hmm_features.columns,
)

volatility_score = state_feature_means[["Abs_Return", "Realized_Volatility", "VIX_Close"]].mean(axis=1)
high_vol_state = int(volatility_score.idxmax())
low_vol_state = 1 - high_vol_state

state_labels = {
    low_vol_state: "low_vol",
    high_vol_state: "high_vol",
}
portfolio_hmm_state_summary = state_feature_means.rename(index=state_labels).loc[["low_vol", "high_vol"]]
portfolio_hmm_transition_matrix = pd.DataFrame(
    portfolio_hmm_model["transmat"],
    index=[state_labels[0], state_labels[1]],
    columns=[state_labels[0], state_labels[1]],
).loc[["low_vol", "high_vol"], ["low_vol", "high_vol"]]

portfolio_hmm_regime_probabilities = pd.DataFrame(
    {
        "Prob_Low_Vol": portfolio_hmm_model["gamma"][:, low_vol_state],
        "Prob_High_Vol": portfolio_hmm_model["gamma"][:, high_vol_state],
    },
    index=portfolio_hmm_features.index,
)
portfolio_hmm_regime_probabilities["Most_Likely_Regime"] = np.where(
    portfolio_hmm_regime_probabilities["Prob_High_Vol"]
    >= portfolio_hmm_regime_probabilities["Prob_Low_Vol"],
    "high_vol",
    "low_vol",
)

data_dir = globals().get("DATA_DIR", Path("data"))
data_dir.mkdir(exist_ok=True)
portfolio_hmm_output = portfolio_hmm_features.join(portfolio_hmm_regime_probabilities)
portfolio_hmm_output.to_csv(data_dir / "portfolio_hmm_regime_probabilities.csv", index_label="Date")

print("Saved: data/portfolio_hmm_regime_probabilities.csv")
print(
    f"HMM converged in {portfolio_hmm_model['iterations']} iterations with log-likelihood "
    f"{portfolio_hmm_model['log_likelihood']:.2f}"
)
portfolio_hmm_state_summary, portfolio_hmm_transition_matrix, portfolio_hmm_output.tail()


Saved: data/portfolio_hmm_regime_probabilities.csv
HMM converged in 22 iterations with log-likelihood -35869.25


(          Portfolio_Return  Abs_Return  Realized_Volatility  VIX_Close
 low_vol           0.000781    0.006459             0.134888  16.231976
 high_vol         -0.000300    0.015850             0.290221  28.616913,
            low_vol  high_vol
 low_vol   0.982393  0.017607
 high_vol  0.050030  0.949970,
             Portfolio_Return  Abs_Return  Realized_Volatility  VIX_Close  \
 Date                                                                       
 2025-12-24          0.004400    0.004400             0.102263      13.47   
 2025-12-26          0.001137    0.001137             0.082288      13.60   
 2025-12-29         -0.005908    0.005908             0.084851      14.20   
 2025-12-30         -0.003279    0.003279             0.086024      14.33   
 2025-12-31         -0.007159    0.007159             0.090103      14.95   
 
             Prob_Low_Vol  Prob_High_Vol Most_Likely_Regime  
 Date                                                        
 2025-12-24      0.999998  

In [10]:
from pathlib import Path


def forward_filter_probabilities(
    startprob: np.ndarray,
    transmat: np.ndarray,
    emission_log_prob: np.ndarray,
) -> tuple[np.ndarray, float]:
    n_obs, n_states = emission_log_prob.shape
    log_startprob = np.log(np.clip(startprob, 1e-12, None))
    log_transmat = np.log(np.clip(transmat, 1e-12, None))

    log_filtered = np.empty((n_obs, n_states))
    log_norm_constants = np.empty(n_obs)

    log_joint = log_startprob + emission_log_prob[0]
    log_norm_constants[0] = logsumexp(log_joint, axis=0)
    log_filtered[0] = log_joint - log_norm_constants[0]

    for t in range(1, n_obs):
        log_pred = logsumexp(log_filtered[t - 1][:, None] + log_transmat, axis=0)
        log_joint = log_pred + emission_log_prob[t]
        log_norm_constants[t] = logsumexp(log_joint, axis=0)
        log_filtered[t] = log_joint - log_norm_constants[t]

    return np.exp(log_filtered), float(log_norm_constants.sum())


def walk_forward_one_step_ahead_forecasts(
    X: np.ndarray,
    initial_filtered_prob: np.ndarray,
    transmat: np.ndarray,
    means: np.ndarray,
    variances: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_obs = X.shape[0]
    n_states = transmat.shape[0]

    current_day_forecast = np.empty((n_obs, n_states))
    filtered_probabilities = np.empty((n_obs, n_states))
    next_day_forecast = np.empty((n_obs, n_states))

    previous_filtered = initial_filtered_prob.copy()

    for t in range(n_obs):
        current_day_forecast[t] = previous_filtered @ transmat
        emission_log_prob_t = gaussian_logpdf_diag(X[t : t + 1], means, variances)[0]
        log_filtered_t = np.log(np.clip(current_day_forecast[t], 1e-12, None)) + emission_log_prob_t
        log_filtered_t -= logsumexp(log_filtered_t, axis=0)
        filtered_probabilities[t] = np.exp(log_filtered_t)
        next_day_forecast[t] = filtered_probabilities[t] @ transmat
        previous_filtered = filtered_probabilities[t]

    return current_day_forecast, filtered_probabilities, next_day_forecast


train_test_split_date = pd.Timestamp("2010-01-01")
data_dir = globals().get("DATA_DIR", Path("data"))
data_dir.mkdir(exist_ok=True)

oos_hmm_features = (
    portfolio_hmm_features.copy()
    if "portfolio_hmm_features" in globals()
    else build_hmm_feature_table(realized_vol_window=21)
)
train_features = oos_hmm_features.loc[oos_hmm_features.index < train_test_split_date].copy()
test_features = oos_hmm_features.loc[oos_hmm_features.index >= train_test_split_date].copy()

if train_features.empty:
    raise ValueError("Training sample is empty. Choose a later split date.")
if test_features.empty:
    raise ValueError("Test sample is empty. Choose an earlier split date.")

train_feature_means = train_features.mean()
train_feature_stds = train_features.std(ddof=0).replace(0, 1.0)
X_train = ((train_features - train_feature_means) / train_feature_stds).to_numpy()
X_test = ((test_features - train_feature_means) / train_feature_stds).to_numpy()

oos_hmm_model = fit_gaussian_hmm(X_train)

train_state_feature_means = pd.DataFrame(
    (
        oos_hmm_model["gamma"].T @ train_features.to_numpy()
    ) / oos_hmm_model["gamma"].sum(axis=0)[:, None],
    index=[0, 1],
    columns=train_features.columns,
)

train_volatility_score = train_state_feature_means[["Abs_Return", "Realized_Volatility", "VIX_Close"]].mean(axis=1)
oos_high_vol_state = int(train_volatility_score.idxmax())
oos_low_vol_state = 1 - oos_high_vol_state

oos_state_labels = {
    oos_low_vol_state: "low_vol",
    oos_high_vol_state: "high_vol",
}
portfolio_hmm_oos_state_summary = train_state_feature_means.rename(index=oos_state_labels).loc[["low_vol", "high_vol"]]
portfolio_hmm_oos_transition_matrix = pd.DataFrame(
    oos_hmm_model["transmat"],
    index=[oos_state_labels[0], oos_state_labels[1]],
    columns=[oos_state_labels[0], oos_state_labels[1]],
).loc[["low_vol", "high_vol"], ["low_vol", "high_vol"]]

train_emission_log_prob = gaussian_logpdf_diag(X_train, oos_hmm_model["means"], oos_hmm_model["variances"])
train_filtered_probabilities, train_filtered_log_likelihood = forward_filter_probabilities(
    oos_hmm_model["startprob"],
    oos_hmm_model["transmat"],
    train_emission_log_prob,
)
last_train_filtered_probability = train_filtered_probabilities[-1]

test_current_day_forecast, test_filtered_probabilities, test_next_day_forecast = walk_forward_one_step_ahead_forecasts(
    X_test,
    last_train_filtered_probability,
    oos_hmm_model["transmat"],
    oos_hmm_model["means"],
    oos_hmm_model["variances"],
)

portfolio_hmm_oos_forecasts = test_features.join(
    pd.DataFrame(
        {
            "Forecast_Prob_Low_Vol": test_current_day_forecast[:, oos_low_vol_state],
            "Forecast_Prob_High_Vol": test_current_day_forecast[:, oos_high_vol_state],
            "Filtered_Prob_Low_Vol": test_filtered_probabilities[:, oos_low_vol_state],
            "Filtered_Prob_High_Vol": test_filtered_probabilities[:, oos_high_vol_state],
            "Next_Day_Forecast_Prob_Low_Vol": test_next_day_forecast[:, oos_low_vol_state],
            "Next_Day_Forecast_Prob_High_Vol": test_next_day_forecast[:, oos_high_vol_state],
        },
        index=test_features.index,
    )
)
portfolio_hmm_oos_forecasts["Forecast_Regime"] = np.where(
    portfolio_hmm_oos_forecasts["Forecast_Prob_High_Vol"]
    >= portfolio_hmm_oos_forecasts["Forecast_Prob_Low_Vol"],
    "high_vol",
    "low_vol",
)
portfolio_hmm_oos_forecasts["Filtered_Regime"] = np.where(
    portfolio_hmm_oos_forecasts["Filtered_Prob_High_Vol"]
    >= portfolio_hmm_oos_forecasts["Filtered_Prob_Low_Vol"],
    "high_vol",
    "low_vol",
)

portfolio_hmm_oos_summary = pd.Series(
    {
        "Train_Start": train_features.index.min().date(),
        "Train_End": train_features.index.max().date(),
        "Test_Start": test_features.index.min().date(),
        "Test_End": test_features.index.max().date(),
        "Train_Observations": len(train_features),
        "Test_Observations": len(test_features),
        "Train_Log_Likelihood": oos_hmm_model["log_likelihood"],
        "Filtered_Train_Log_Likelihood": train_filtered_log_likelihood,
    }
)

portfolio_hmm_oos_forecasts.to_csv(
    data_dir / "portfolio_hmm_oos_2010_forecasts.csv",
    index_label="Date",
)

print("Saved: data/portfolio_hmm_oos_2010_forecasts.csv")
print(
    f"Train sample: {train_features.index.min().date()} to {train_features.index.max().date()} | "
    f"Test sample: {test_features.index.min().date()} to {test_features.index.max().date()}"
)
portfolio_hmm_oos_summary, portfolio_hmm_oos_state_summary, portfolio_hmm_oos_transition_matrix, portfolio_hmm_oos_forecasts.head()


Saved: data/portfolio_hmm_oos_2010_forecasts.csv
Train sample: 1990-01-02 to 2009-12-31 | Test sample: 2010-01-04 to 2025-12-31


(Train_Start                        1990-01-02
 Train_End                          2009-12-31
 Test_Start                         2010-01-04
 Test_End                           2025-12-31
 Train_Observations                       5043
 Test_Observations                        4024
 Train_Log_Likelihood            -20034.040383
 Filtered_Train_Log_Likelihood   -20034.040368
 dtype: object,
           Portfolio_Return  Abs_Return  Realized_Volatility  VIX_Close
 low_vol           0.000672    0.006764             0.138872  17.048293
 high_vol         -0.000165    0.016626             0.307113  29.933636,
            low_vol  high_vol
 low_vol   0.985829  0.014171
 high_vol  0.042186  0.957814,
             Portfolio_Return  Abs_Return  Realized_Volatility  VIX_Close  \
 Date                                                                       
 2010-01-04          0.012637    0.012637             0.125532  20.040001   
 2010-01-05         -0.002100    0.002100             0.120249  19.35

In [11]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from pathlib import Path
from plotly.subplots import make_subplots


def load_threshold_regime_table() -> pd.DataFrame:
    data_dir = globals().get("DATA_DIR", Path("data"))
    if "vix_regimes" in globals():
        threshold_df = vix_regimes.copy()
    else:
        threshold_df = pd.read_csv(
            data_dir / "vix_regime_classification.csv",
            index_col="Date",
            parse_dates=True,
        )

    threshold_df = threshold_df.copy()
    threshold_df["Threshold_High_Vol"] = (threshold_df["Regime"] == "high_vol").astype(float)
    return threshold_df


def load_hmm_output_table() -> pd.DataFrame:
    data_dir = globals().get("DATA_DIR", Path("data"))
    if "portfolio_hmm_oos_forecasts" in globals():
        hmm_df = portfolio_hmm_oos_forecasts.copy()
    else:
        hmm_df = pd.read_csv(
            data_dir / "portfolio_hmm_oos_2010_forecasts.csv",
            index_col="Date",
            parse_dates=True,
        )

    required_columns = [
        "Forecast_Prob_High_Vol",
        "Filtered_Prob_High_Vol",
        "Realized_Volatility",
    ]
    missing_columns = [col for col in required_columns if col not in hmm_df.columns]
    if missing_columns:
        raise ValueError(f"Missing OOS HMM output columns: {missing_columns}")

    hmm_df = hmm_df.copy()
    hmm_df["Forecast_Prob_Low_Vol"] = 1.0 - hmm_df["Forecast_Prob_High_Vol"]
    hmm_df["Filtered_Prob_Low_Vol"] = 1.0 - hmm_df["Filtered_Prob_High_Vol"]
    return hmm_df


def get_high_vol_intervals(high_vol_state: pd.Series) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    intervals = []
    in_interval = False
    interval_start = None
    previous_date = None

    for current_date, is_high_vol in high_vol_state.astype(bool).items():
        if is_high_vol and not in_interval:
            interval_start = current_date
            in_interval = True
        elif not is_high_vol and in_interval:
            intervals.append((interval_start, previous_date + pd.Timedelta(days=1)))
            in_interval = False

        previous_date = current_date

    if in_interval:
        intervals.append((interval_start, previous_date + pd.Timedelta(days=1)))

    return intervals


oos_start_date = pd.Timestamp(globals().get("train_test_split_date", "2010-01-01"))
threshold_regimes = load_threshold_regime_table()
hmm_output = load_hmm_output_table()

comparison_df = hmm_output[
    [
        "Forecast_Prob_Low_Vol",
        "Forecast_Prob_High_Vol",
        "Filtered_Prob_Low_Vol",
        "Filtered_Prob_High_Vol",
        "Realized_Volatility",
    ]
].join(
    threshold_regimes[["VIX_Close", "Threshold_High_Vol", "Regime"]],
    how="inner",
)
comparison_df = comparison_df.loc[comparison_df.index >= oos_start_date].copy()
if comparison_df.empty:
    raise ValueError(
        "No overlapping dates between the OOS HMM forecasts and the VIX threshold regimes "
        "after the train/test split date."
    )

comparison_df["Forecast_HMM_High_Vol_State"] = (
    comparison_df["Forecast_Prob_High_Vol"] >= comparison_df["Forecast_Prob_Low_Vol"]
).astype(float)
comparison_df["Filtered_HMM_High_Vol_State"] = (
    comparison_df["Filtered_Prob_High_Vol"] >= comparison_df["Filtered_Prob_Low_Vol"]
).astype(float)

comparison_summary = pd.Series(
    {
        "Sample_Start": comparison_df.index.min().date(),
        "Sample_End": comparison_df.index.max().date(),
        "Threshold_High_Vol_Share": comparison_df["Threshold_High_Vol"].mean(),
        "Forecast_Avg_Prob_High_Vol": comparison_df["Forecast_Prob_High_Vol"].mean(),
        "Filtered_Avg_Prob_High_Vol": comparison_df["Filtered_Prob_High_Vol"].mean(),
        "Forecast_Binary_Agreement": (
            comparison_df["Threshold_High_Vol"] == comparison_df["Forecast_HMM_High_Vol_State"]
        ).mean(),
        "Filtered_Binary_Agreement": (
            comparison_df["Threshold_High_Vol"] == comparison_df["Filtered_HMM_High_Vol_State"]
        ).mean(),
    }
)


In [12]:

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    row_heights=[0.35, 0.65],
    specs=[[{"secondary_y": True}], [{"secondary_y": False}]],
)
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    row_heights=[0.35, 0.65],
    specs=[[{"secondary_y": True}], [{"secondary_y": False}]],
)

fig.add_trace(
    go.Scatter(
        x=comparison_df.index,
        y=comparison_df["VIX_Close"],
        mode="lines",
        name="VIX Close",
        line=dict(color="#111111", width=1.6),
        hovertemplate="Date=%{x|%Y-%m-%d}<br>VIX=%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=comparison_df.index,
        y=comparison_df["Realized_Volatility"],
        mode="lines",
        name="Realized Volatility",
        line=dict(color="#2E8B57", width=1.6),
        hovertemplate="Date=%{x|%Y-%m-%d}<br>Realized vol=%{y:.3f}<extra></extra>",
    ),
    row=1,
    col=1,
    secondary_y=True,
)

fig.add_trace(
    go.Scatter(
        x=comparison_df.index,
        y=comparison_df["Forecast_Prob_High_Vol"],
        mode="lines",
        line=dict(color="rgba(200, 30, 45, 0)", width=0),
        fill="tozeroy",
        fillcolor="rgba(200, 30, 45, 0.48)",
        name="Forecast High-Vol Area",
        hoverinfo="skip",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=comparison_df.index,
        y=np.ones(len(comparison_df)),
        mode="lines",
        line=dict(color="rgba(45, 95, 210, 0)", width=0),
        fill="tonexty",
        fillcolor="rgba(45, 95, 210, 0.35)",
        name="Forecast Low-Vol Area",
        hoverinfo="skip",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=comparison_df.index,
        y=comparison_df["Forecast_Prob_High_Vol"],
        mode="lines",
        name="One-Day-Ahead High-Vol Forecast",
        line=dict(color="#8B0000", width=1.5),
        customdata=np.column_stack(
            [
                comparison_df["Forecast_Prob_Low_Vol"],
                comparison_df["Filtered_Prob_High_Vol"],
                comparison_df["Threshold_High_Vol"],
                comparison_df["Regime"],
            ]
        ),
        hovertemplate=(
            "Date=%{x|%Y-%m-%d}<br>Forecast Prob High Vol=%{y:.3f}"
            "<br>Forecast Prob Low Vol=%{customdata[0]:.3f}"
            "<br>Filtered Prob High Vol=%{customdata[1]:.3f}"
            "<br>VIX Threshold High Vol=%{customdata[2]:.0f}"
            "<br>Threshold Regime=%{customdata[3]}<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=comparison_df.index,
        y=comparison_df["Filtered_Prob_High_Vol"],
        mode="lines",
        name="Same-Day Filtered High-Vol Probability",
        line=dict(color="#1E5AA8", width=1.2, dash="dash"),
        customdata=np.column_stack(
            [
                comparison_df["Filtered_Prob_Low_Vol"],
                comparison_df["Forecast_Prob_High_Vol"],
                comparison_df["Threshold_High_Vol"],
                comparison_df["Regime"],
            ]
        ),
        hovertemplate=(
            "Date=%{x|%Y-%m-%d}<br>Filtered Prob High Vol=%{y:.3f}"
            "<br>Filtered Prob Low Vol=%{customdata[0]:.3f}"
            "<br>Forecast Prob High Vol=%{customdata[1]:.3f}"
            "<br>VIX Threshold High Vol=%{customdata[2]:.0f}"
            "<br>Threshold Regime=%{customdata[3]}<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="rgba(0, 0, 0, 0.35)", width=10),
        name="VIX Threshold High-Vol Period",
        hoverinfo="skip",
    ),
    row=2,
    col=1,
)

for interval_start, interval_end in get_high_vol_intervals(comparison_df["Threshold_High_Vol"]):
    fig.add_vrect(
        x0=interval_start,
        x1=interval_end,
        fillcolor="rgba(0, 0, 0, 0.12)",
        line_width=0,
        layer="above",
        row=2,
        col=1,
    )

fig.update_yaxes(title_text="VIX Close", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Realized Volatility", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="High-Vol Probability", range=[0, 1], row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1, rangeslider_visible=True)

fig.update_layout(
    title="Out-of-Sample Regime Comparison: VIX Threshold vs HMM Forecast (2010+)",
    height=850,
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=60, r=60, t=80, b=60),
)

fig.show()

comparison_summary, comparison_df[[
    "Regime",
    "Forecast_Prob_High_Vol",
    "Filtered_Prob_High_Vol",
    "VIX_Close",
    "Realized_Volatility",
]].tail()


(Sample_Start                  2010-01-04
 Sample_End                    2025-12-31
 Threshold_High_Vol_Share         0.21173
 Forecast_Avg_Prob_High_Vol      0.212895
 Filtered_Avg_Prob_High_Vol      0.210592
 Forecast_Binary_Agreement       0.718191
 Filtered_Binary_Agreement        0.73161
 dtype: object,
              Regime  Forecast_Prob_High_Vol  Filtered_Prob_High_Vol  \
 Date                                                                  
 2025-12-24  low_vol                0.014221                0.000038   
 2025-12-26  low_vol                0.014207                0.000083   
 2025-12-29  low_vol                0.014250                0.000075   
 2025-12-30  low_vol                0.014242                0.000065   
 2025-12-31  low_vol                0.014233                0.000075   
 
             VIX_Close  Realized_Volatility  
 Date                                        
 2025-12-24      13.47             0.102263  
 2025-12-26      13.60             0.082288  
